# ABSim — Tutorial

ABSim is a Couzin-style agent-based simulation with three social interaction zones (repulsion, orientation, attraction) and support for multiple agent types.

Install once if needed:
```bash
pip install -e .
```

## 1 — Launch the default simulation

Runs a prey–predator world with all the default parameters.  
The window opens interactively; **the kernel is blocked until you close it.**

In [ ]:
import ABSim

ABSim.run()

## 2 — Custom agent types

Pass a list of `AgentType` objects to `run()` to define your own scenario.  
Each type has:

| Parameter | What it controls |
|---|---|
| `color` | Triangle fill colour (any Tk colour string) |
| `size` | Body length in pixels |
| `speed_scale` | Multiplier on the global Speed slider |
| `max_turn` | Maximum heading change per step (radians) |
| `wander_noise` | Random heading jitter when isolated (fraction of max_turn) |
| `hunts` | Type names this type chases and catches |
| `flees` | Type names this type runs away from |

The **N Prey** slider always controls the first type; **N Predators** controls the second.

In [ ]:
# --- Example A: classic predator–prey ---
# Same as the default, but with explicit parameters you can tweak.
import ABSim, math

prey = ABSim.AgentType(
    "prey",
    color="#56b1fc",
    size=15,
    speed_scale=1.0,
    max_turn=math.pi * 0.2,
    flees=("predator",),
)

predator = ABSim.AgentType(
    "predator",
    color="#fc6056",
    size=20,
    speed_scale=1.5,       # predators move 50% faster than the Speed slider value
    max_turn=math.pi * 0.15,
    wander_noise=0.3,      # wander actively when no prey in sight
    hunts=("prey",),
)

ABSim.run(agent_types=[prey, predator])

In [ ]:
# --- Example B: fast shoal vs slow shoal ---
# Two groups with the same social rules, different speeds.
# Neither hunts the other — they just coexist.
import ABSim

slow_shoal = ABSim.AgentType("slow", color="#56b1fc", size=14, speed_scale=0.5)
fast_shoal = ABSim.AgentType("fast", color="#f4a030", size=14, speed_scale=2.0)

ABSim.run(agent_types=[slow_shoal, fast_shoal])

In [ ]:
# --- Example C: two independent shoals ---
# Same speed, different colours.  Useful for studying group mixing.
import ABSim

shoal_a = ABSim.AgentType("shoal_a", color="#56b1fc", size=14)
shoal_b = ABSim.AgentType("shoal_b", color="#a060f0", size=14)

ABSim.run(agent_types=[shoal_a, shoal_b])

In [ ]:
# --- Example D: timid vs bold ---
# Both types flee predators, but the bold group turns more slowly
# and has a wider attraction zone (set via sliders at runtime).
import ABSim, math

timid = ABSim.AgentType(
    "timid",
    color="#56b1fc",
    size=13,
    max_turn=math.pi * 0.3,   # turns fast — more reactive
    speed_scale=1.2,
    flees=("predator",),
)

bold = ABSim.AgentType(
    "bold",
    color="#40c090",
    size=16,
    max_turn=math.pi * 0.1,   # turns slowly — less reactive
    speed_scale=1.0,
    flees=("predator",),
)

predator = ABSim.AgentType(
    "predator",
    color="#fc6056",
    size=20,
    speed_scale=1.5,
    hunts=("timid", "bold"),  # hunts both
)

ABSim.run(agent_types=[timid, bold, predator])

## 3 — Headless simulation

Call `model.step()` in a loop to collect data without opening the GUI.  
The `step()` arguments mirror the GUI sliders:

| Argument | Slider it matches |
|---|---|
| `r_r` | ZOR (repulsion zone radius) |
| `r_o` | ZOO (orientation zone outer radius) |
| `r_a` | ZOA (attraction zone outer radius) |
| `flee_dist` | Flee dist (predator) |
| `catch_dist` | Catch distance |
| `desired_speed` | Speed |
| `var_speed` | Variable speed checkbox |
| `fov_deg` | Field of view (°) |

In [ ]:
import ABSim

N_STEPS = 500
N_PREY  = 100
N_PREDS = 5

# Zone radii and flee_dist are now per-type attributes on AgentType.
# Set them before or after creating the model — they're read live each step.
prey      = ABSim.AgentType("prey",     r_r=10, r_o=30, r_a=75,  flee_dist=100,
                             flees=("predator",), n_default=N_PREY)
predator  = ABSim.AgentType("predator", r_r=10, r_o=50, r_a=150, flee_dist=100,
                             speed_scale=1.5, hunts=("prey",), n_default=N_PREDS)

model = ABSim.Model(agent_types=[prey, predator])
model.init_agents("prey",     N_PREY)
model.init_agents("predator", N_PREDS)

prey_counts = [len(model.groups["prey"])]

for _ in range(N_STEPS):
    model.step(catch_dist=40, desired_speed=3)
    prey_counts.append(len(model.groups["prey"]))

print(f"Start: {prey_counts[0]} prey")
print(f"End ({N_STEPS} steps): {prey_counts[-1]} prey")
print(f"Survival: {prey_counts[-1] / N_PREY:.0%}")

In [ ]:
# Plot survival over time (requires matplotlib)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(prey_counts, color="#56b1fc")
plt.xlabel("Step")
plt.ylabel("Prey alive")
plt.title(f"Prey survival — {N_PREDS} predators, {N_STEPS} steps")
plt.tight_layout()
plt.show()

In [ ]:
import ABSim

N_STEPS = 300

scenarios = {
    "small ZOA (r_a=40)":  dict(r_r=10, r_o=25, r_a=40),
    "medium ZOA (r_a=75)": dict(r_r=10, r_o=30, r_a=75),
    "large ZOA (r_a=150)": dict(r_r=10, r_o=40, r_a=150),
}

for label, zones in scenarios.items():
    prey     = ABSim.AgentType("prey",     **zones, flee_dist=100,
                                flees=("predator",), n_default=100)
    predator = ABSim.AgentType("predator", r_r=10, r_o=50, r_a=150,
                                speed_scale=1.5, hunts=("prey",), n_default=5)
    m = ABSim.Model(agent_types=[prey, predator])
    m.init_agents("prey",     100)
    m.init_agents("predator", 5)
    for _ in range(N_STEPS):
        m.step(catch_dist=40, desired_speed=3)
    print(f"{label}: {len(m.groups['prey'])}/100 prey survived")